In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load Data
file_path = '/content/cleaned_1min_master_scrubbed.parquet'
df_master = pd.read_parquet(file_path)

# 2. Select a subset for clean visualizations (broad market proxies)
sample_etfs = ['RSP', 'DIA', 'SPY', 'QQQ']
df_sample = df_master[sample_etfs].copy()

# 3. Transformations
df_log_prices = np.log(df_sample)
df_log_returns = df_log_prices.diff().dropna()
df_normalized = df_sample / df_sample.iloc[0] # Growth of $1.00

# 4. Plotting
fig, axes = plt.subplots(4, 1, figsize=(14, 16))

# A. Raw Prices
df_sample.plot(ax=axes[0], alpha=0.9)
axes[0].set_title('Raw 1-Minute Prices', fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

# B. Normalized Prices
df_normalized.plot(ax=axes[1], alpha=0.9)
axes[1].set_title('Normalized Prices (Base = 1.0)', fontweight='bold')
axes[1].set_ylabel('Growth Scale')
axes[1].grid(True, alpha=0.3)

# C. Log Prices
df_log_prices.plot(ax=axes[2], alpha=0.9)
axes[2].set_title('Log Prices', fontweight='bold')
axes[2].set_ylabel('Log(Price)')
axes[2].grid(True, alpha=0.3)

# D. Log Returns
df_log_returns.plot(ax=axes[3], alpha=0.6, linewidth=0.5)
axes[3].set_title('1-Minute Log Returns', fontweight='bold')
axes[3].set_ylabel('Log Return')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 5. Data Integrity Checks (Run on ALL 34 ETFs)
print("--- Global Data Integrity Check ---")

# Check for extreme 1-minute returns (Outliers/Bad Prints)
# A >2% move in a broad ETF in a single minute is highly suspicious
all_log_returns = np.log(df_master).diff().dropna()
threshold = 0.02
outliers = (all_log_returns.abs() > threshold).sum()

print("\n1. Outliers Detected (>2% move in 1 minute):")
if outliers.sum() == 0:
    print("   ✓ No extreme outliers found. Price continuity looks clean.")
else:
    print(outliers[outliers > 0])

# Check for Intraday Gaps
# Diffs > 1 min but < 12 hours (to ignore standard overnight gaps)
time_diffs = df_master.index.to_series().diff()
intraday_gaps = time_diffs[(time_diffs > pd.Timedelta(minutes=1)) & (time_diffs < pd.Timedelta(hours=12))]

print("\n2. Missing Intraday Bars:")
if intraday_gaps.empty:
    print("   ✓ No intraday gaps. 1-minute timeline is perfectly continuous during market hours.")
else:
    print(f"   ⚠️ Found {len(intraday_gaps)} instances of missing 1-minute bars during RTH.")
    print(intraday_gaps.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/cleaned_1min_master_scrubbed.parquet'

In [ ]:
import numpy as np

# 1. Create a copy of the master data
df_scrubbed = df_master.copy()

# 2. Define the maximum allowable 1-minute move (5%)
spike_threshold = 0.05

print("Scrubbing bad prints...")

for col in df_scrubbed.columns:
    # Skip VIX since it naturally has massive 1-minute swings
    if col == 'VIX':
        continue

    # Calculate absolute percentage returns
    pct_rets = df_scrubbed[col].pct_change().abs()

    # Count how many bad ticks we are fixing for this specific ETF
    bad_tick_count = (pct_rets > spike_threshold).sum()
    if bad_tick_count > 0:
        print(f"Fixed {bad_tick_count} bad prints in {col}")

    # Replace bad prints with NaN
    bad_ticks_mask = pct_rets > spike_threshold
    df_scrubbed.loc[bad_ticks_mask, col] = np.nan

# 3. Forward-fill the NaNs with the last legitimate price
df_scrubbed.ffill(inplace=True)

# 4. Quick visual confirmation on RSP
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
df_scrubbed['RSP'].pct_change().plot(color='blue', alpha=0.7)
plt.title('RSP 1-Minute Returns (Post-Scrubbing)', fontweight='bold')
plt.ylabel('Percent Return')
plt.grid(True, alpha=0.3)
plt.show()

# 5. Overwrite the old master with the clean one
df_master = df_scrubbed.copy()

In [ ]:
import numpy as np

# 1. Create a copy of the master data
df_scrubbed_2 = df_master.copy()
df_scrubbed_2.head(5)

# # 2. Define the maximum allowable 1-minute move (5%)
# spike_threshold = 0.05

# print("Scrubbing bad prints...")

# for col in df_scrubbed_2.columns:
#     # Skip VIX since it naturally has massive 1-minute swings
#     if col == 'VIX':
#         continue

#     # Calculate absolute percentage returns
#     pct_rets = df_scrubbed[col].pct_change().abs()

#     # Count how many bad ticks we are fixing for this specific ETF
#     bad_tick_count = (pct_rets > spike_threshold).sum()
    # if bad_tick_count > 0:
    #     print(f"Fixed {bad_tick_count} bad prints in {col}")

    # # Replace bad prints with NaN
    # bad_ticks_mask = pct_rets > spike_threshold
    # df_scrubbed.loc[bad_ticks_mask, col] = np.nan

# # 3. Forward-fill the NaNs with the last legitimate price
# df_scrubbed.ffill(inplace=True)

# # 4. Quick visual confirmation on RSP
# import matplotlib.pyplot as plt

# plt.figure(figsize=(12, 4))
# df_scrubbed['RSP'].pct_change().plot(color='blue', alpha=0.7)
# plt.title('RSP 1-Minute Returns (Post-Scrubbing)', fontweight='bold')
# plt.ylabel('Percent Return')
# plt.grid(True, alpha=0.3)
# plt.show()

# # 5. Overwrite the old master with the clean one
# df_master = df_scrubbed.copy()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_clean = df_master.copy()

print("Applying Centered Rolling Median Filter...")

for col in df_clean.columns:
    if col == 'VIX':
        continue

    # 1. Calculate a 15-minute rolling median, centered on the current bar
    rolling_med = df_clean[col].rolling(window=15, center=True, min_periods=1).median()

    # 2. Calculate how far the actual price deviates from this local median
    dev = (df_clean[col] - rolling_med) / rolling_med

    # 3. Flag anything > 2% away from the neighborhood as a bad print
    bad_mask = dev.abs() > 0.02

    bad_count = bad_mask.sum()
    if bad_count > 0:
        print(f"Scrubbed {bad_count} persistent outliers in {col}")
        # Replace the bad print with the local median price
        df_clean.loc[bad_mask, col] = rolling_med[bad_mask]

# Update the master dataset
df_master = df_clean.copy()

# Plot to verify we finally killed the 40% spikes
plt.figure(figsize=(12, 4))
df_master['RSP'].pct_change().plot(color='darkgreen', alpha=0.7, linewidth=0.5)
plt.title('RSP 1-Minute Returns (Post-Median Scrubbing)', fontweight='bold')
plt.ylabel('Percent Return')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Save locally inside the Colab session environment
local_save_path = '/content/cleaned_1min_master_scrubbed.parquet'

print("Exporting scrubbed dataset locally...")
df_master.to_parquet(local_save_path)

print(f"✅ File successfully saved to: {local_save_path}")

In [ ]:
import pandas as pd
import numpy as np
import itertools
from statsmodels.tsa.stattools import coint

print("Resampling to Daily bars for efficient cointegration sweep...")
# Resample to daily closing prices and drop NaNs to align time series perfectly
df_daily = df_master.resample('D').last().dropna(how='all')
# Forward fill any single-day missing values, then drop remaining NAs
df_daily = df_daily.ffill().dropna()

# Get all tradable tickers (exclude VIX)
tickers = [col for col in df_daily.columns if col != 'VIX']

# Generate all unique pairs
pairs = list(itertools.combinations(tickers, 2))
print(f"Testing {len(pairs)} unique ETF combinations...")

results = []

for pair in pairs:
    y = df_daily[pair[0]]
    x = df_daily[pair[1]]

    # Run Engle-Granger test for cointegration
    # coint returns: t-statistic, p-value, critical values
    try:
        score, p_value, _ = coint(y, x)
        results.append({
            'Leg 1': pair[0],
            'Leg 2': pair[1],
            'p_value': p_value,
            't_stat': score
        })
    except Exception as e:
        continue

# Convert to DataFrame and sort by strongest cointegration (lowest p-value)
df_coint = pd.DataFrame(results)
df_coint = df_coint.sort_values(by='p_value')
df_coint['p_value'] = df_coint['p_value'].round(5)
df_coint['t_stat'] = df_coint['t_stat'].round(3)

print("\n--- Top 15 Most Cointegrated ETF Pairs ---")
print(df_coint.head(15).to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings('ignore')

# 1. Select key ETFs for analysis
selected_tickers = ['XLV', 'XLY', 'XLU', 'RSP', 'DIA']
df_selected = df_master[selected_tickers].copy()

# Resample to Daily for clean macro stationarity verification
df_daily = df_selected.resample('D').last().ffill().dropna()

results = []

def run_stationarity_tests(series, name, transform_type):
    # ADF Test
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series)
    adf_stationary = adf_p < 0.05

    # KPSS Test
    kpss_stat, kpss_p, _, kpss_crit = kpss(series, regression='c', nlags='auto')
    kpss_stationary = kpss_p >= 0.05

    return {
        'Ticker': name,
        'Transformation': transform_type,
        'ADF p-val': round(adf_p, 4),
        'ADF Stationary?': 'Yes' if adf_stationary else 'No',
        'KPSS p-val': round(kpss_p, 4),
        'KPSS Stationary?': 'Yes' if kpss_stationary else 'No',
        'Overall Conclusion': 'Stationary' if (adf_stationary and kpss_stationary) else 'Non-Stationary'
    }

# 2. Iterate across Raw Prices, Log Prices, and Log Returns
for ticker in selected_tickers:
    raw_prices = df_daily[ticker]
    log_prices = np.log(raw_prices)
    log_returns = log_prices.diff().dropna()

    results.append(run_stationarity_tests(raw_prices, ticker, 'Raw Price'))
    results.append(run_stationarity_tests(log_prices, ticker, 'Log Price'))
    results.append(run_stationarity_tests(log_returns, ticker, 'Log Return'))

# 3. Display summary
df_stationarity = pd.DataFrame(results)
print("--- Stationarity Test Results (ADF & KPSS) ---")
print(df_stationarity.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 1. Load ONLY the required tickers to keep memory footprint minimal
required_tickers = ['XLV', 'XLY', 'XLU', 'RSPM', 'RSP', 'DIA']
file_path = '/content/cleaned_1min_master_scrubbed.parquet'

df_sub = pd.read_parquet(file_path, columns=required_tickers)

# 2. Resample selected series to 5-minute bars
df_5min = df_sub.resample('5min').last().ffill().dropna()

candidate_pairs = [
    ('XLV', 'XLY'),
    ('XLU', 'XLY'),
    ('RSPM', 'XLV'),
    ('RSP', 'DIA')
]

metrics = []
spreads_dict = {}

for leg1, leg2 in candidate_pairs:
    log_y = np.log(df_5min[leg1].values)
    log_x = np.log(df_5min[leg2].values)

    # Fast NumPy OLS for Hedge Ratio (Beta)
    beta, alpha = np.polyfit(log_x, log_y, 1)

    # Construct 5-minute residual spread
    spread_5min = log_y - (beta * log_x + alpha)
    spreads_dict[f"{leg1}/{leg2}"] = pd.Series(spread_5min, index=df_5min.index)

    # Fast NumPy OLS for Ornstein-Uhlenbeck process (Half-Life)
    spread_lag = spread_5min[:-1]
    spread_diff = np.diff(spread_5min)
    X_ou = np.column_stack([np.ones_like(spread_lag), spread_lag])
    params = np.linalg.lstsq(X_ou, spread_diff, rcond=None)[0]
    lambda_param = params[1]

    hl_bars = -np.log(2) / lambda_param if lambda_param < 0 else np.nan
    hl_days = hl_bars / 78 if not np.isnan(hl_bars) else np.nan  # 78 five-minute bars per day

    # Resample spread to Daily for safe ADF/KPSS stationarity testing
    spread_daily = spreads_dict[f"{leg1}/{leg2}"].resample('D').last().dropna()
    adf_p = adfuller(spread_daily)[1]
    kpss_p = kpss(spread_daily, regression='c', nlags='auto')[1]

    metrics.append({
        'Pair': f"{leg1} / {leg2}",
        'Hedge Ratio (β)': round(beta, 4),
        'Spread ADF p-val': round(adf_p, 5),
        'Spread KPSS p-val': round(kpss_p, 4),
        'Half-Life (Trading Days)': round(hl_days, 2) if not np.isnan(hl_days) else "No Reversion"
    })

# 3. Output Analysis Summary Table
df_summary = pd.DataFrame(metrics)
print("--- Cointegration & Spread Stationarity Summary ---")
print(df_summary.to_string(index=False))

# 4. Plot Standardized Spread History (Z-Scores)
plt.figure(figsize=(14, 6))
for pair_name, spread_series in spreads_dict.items():
    z_score = (spread_series - spread_series.mean()) / spread_series.std()
    plt.plot(z_score, label=pair_name, alpha=0.65, linewidth=0.7)

plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.axhline(2.0, color='red', linestyle=':', label='+2σ Entry')
plt.axhline(-2.0, color='green', linestyle=':', label='-2σ Entry')
plt.title('Standardized Spread History (5-Min Z-Scores)', fontweight='bold')
plt.ylabel('Z-Score')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.show()